In [ ]:
%%sql -r dataframe_1
USE ROLE DOG_DATA5035_ROLE;

# Exercise 12: Feature Engineering

## Overview
Building on the MOAB rover Ra-226 survey dataset, this notebook:
1. Completes the lab work (coordinate conversion, layered averages, reference-range comparison, AI prioritization)
2. Adds **three additional engineered features**:

| # | Category | Feature | Description |
|---|----------|---------|-------------|
| 1 | Assess   | Z-Score vs Global Mean | How many standard deviations each tile's average is from the dataset-wide mean |
| 2 | Combine  | Sensor Variability Index | Coefficient of variation (stddev / avg) — normalizes variability across tiles |
| 3 | Rank     | Local Rank Within Region | Ranks each survey unit by its average reading within its parent tile |

## SCALARS Mapping
- **Simplify**: convert raw coordinates into tile / SU / subcell IDs
- **Aggregate**: summarize measurements at each grid level
- **Assess**: compare readings to expected range + Z-score
- **Combine**: sensor variability index (CV)
- **Label/Rank**: local rank of survey units within their tile

In [ ]:
%%sql -r source_data
-- Change to your user's schema
USE SCHEMA data5035.DOG;
SELECT * FROM data5035.spring26.sdg_001_ra226_scandata LIMIT 10;

## Part 1 — Convert Coordinates to Grid

Raw easting/northing values are converted to a three-level grid:
- **Tile** — largest unit, labeled with two letters (e.g. `AA`)
- **Survey Unit (SU)** — 32.81 ft × 32.81 ft cell within a tile, numbered top-left to bottom-right
- **Subcell** — 10×10 subdivision within an SU, numbered bottom-left to top-right

In [ ]:
%%sql -r dataframe_2
CREATE OR REPLACE FUNCTION CONVERT_XY(
        X FLOAT,
        Y FLOAT,
        ORIGIN_X FLOAT,
        ORIGIN_Y FLOAT,
        SU_SIZE FLOAT,
        TILE_GRID_X NUMBER(38,0),
        TILE_GRID_Y NUMBER(38,0),
        SUBCELL_GRID NUMBER(38,0)
    )
    RETURNS OBJECT
    LANGUAGE PYTHON
    RUNTIME_VERSION = '3.11'
    HANDLER = 'convert_xy'
    AS 
    $$
import math

def convert_xy(x, y, origin_x, origin_y, su_size, tile_grid_x, tile_grid_y, subcell_grid):
    # --- Step 1: Compute offset from the known map origin (in feet) ---
    dx = x - origin_x   # horizontal distance from origin
    dy = y - origin_y   # vertical distance from origin

    # --- Step 2: Convert feet to fractional Survey Unit coordinates ---
    su_x = dx / su_size
    su_y = dy / su_size

    # --- Step 3: Determine which Tile the point falls in ---
    tile_col = int(math.floor(su_x / tile_grid_x))  # 0-indexed column (left to right)
    tile_row = int(math.floor(su_y / tile_grid_y))  # 0-indexed row (bottom to top)

    # --- Step 4: Determine local SU position within the tile ---
    local_su_col = int(math.floor(su_x - tile_col * tile_grid_x))
    local_su_row = int(math.floor(su_y - tile_row * tile_grid_y))

    # SU numbering reads left-to-right, top-to-bottom (like a page)
    local_su_row_from_top = (tile_grid_y - 1) - local_su_row
    su_number = local_su_row_from_top * tile_grid_x + local_su_col + 1  # 1-indexed

    # --- Step 5: Determine Subcell within the SU ---
    frac_x = (su_x - math.floor(su_x)) * subcell_grid
    frac_y = (su_y - math.floor(su_y)) * subcell_grid
    subcell_col = int(math.floor(frac_x))
    subcell_row = int(math.floor(frac_y))
    # Subcells number bottom-left to top-right
    subcell = subcell_row * subcell_grid + subcell_col + 1  # 1-indexed

    # --- Step 6: Convert tile indices to two-letter label ---
    row_letter = chr(ord('A') + tile_row)
    col_letter = chr(ord('A') + tile_col)
    tile_label = row_letter + col_letter

    return {"tile": tile_label, "su": su_number, "subcell": subcell}
    $$;

In [ ]:
%%sql -r dataframe_3
-- Convenience overload with MOAB survey's fixed parameters baked in.
CREATE OR REPLACE FUNCTION CONVERT_XY(X FLOAT, Y FLOAT)
    RETURNS OBJECT
    LANGUAGE SQL
    AS
    $$
        SELECT CONVERT_XY(
            X, Y,
            2180160.0001,   -- origin easting (US feet)
            6660000.0000,   -- origin northing (US feet)
            32.81,          -- survey unit size in feet
            21,             -- tile width in survey units
            18,             -- tile height in survey units
            10              -- subcell grid dimension (10x10 per SU)
        )
    $$;

In [ ]:
%%sql -r dataframe_4
-- Test 1: origin point -> expect tile=AA, su=358, subcell=1
SELECT convert_xy(2180160.0001, 6660000.0000);

In [ ]:
%%sql -r dataframe_5
-- Test 2: one full tile-width to the right -> expect tile=AB, su=358, subcell=1
SELECT convert_xy(2180160.0001 + 32.81*21.01, 6660000.0000);

In [ ]:
%%sql -r dataframe_6
-- Test 3: one tile right + one tile up + small offset -> expect tile=BB, su=338, subcell=12
SELECT convert_xy(2180160.0001 + 32.81*22.01 + 4, 6660000.0000 + 32.81*19.01 + 4);

In [ ]:
%%sql -r dataframe_7
-- Apply CONVERT_XY to every scan point and unpack the JSON object into
-- separate typed columns alongside all original fields.
SELECT 
    convert_xy(easting, northing) AS coordinates,
    coordinates:su::INTEGER       AS su,
    coordinates:subcell::INTEGER  AS subcell,
    coordinates:tile::STRING      AS tile,
    * 
FROM data5035.spring26.sdg_001_ra226_scandata 
LIMIT 100;

## Part 2 — Compute Layered Averages

Aggregate all raw readings down to the finest grain: tile > survey unit > subcell.

In [ ]:
%%sql -r dataframe_8
-- Average all raw readings at the tile > SU > subcell grain.
-- GROUP BY ALL collapses every unique coordinate combination.
SELECT
    convert_xy(easting, northing) AS coordinates,
    coordinates:tile::STRING      AS tile,
    coordinates:su::INTEGER       AS su,
    coordinates:subcell::INTEGER  AS subcell,
    AVG(reading)                  AS avg_reading
FROM data5035.spring26.sdg_001_ra226_scandata
GROUP BY ALL
ORDER BY 2, 3, 4;

## Part 3 — Compare to Reference Ranges

Ra-226 action thresholds (pCi/g):
- `< 5` → **OK**
- `5 ≤ x < 7.4` → **Warning**
- `≥ 7.4` → **Alarm**

In [ ]:
%%sql -r dataframe_9
-- Classify each subcell's average reading against Ra-226 thresholds,
-- then roll up counts per tile so the most critical areas surface first.

WITH subcell_avgs AS (
    -- Step 1: Average all raw readings down to the subcell grain
    SELECT
        convert_xy(easting, northing)    AS coordinates,
        coordinates:tile::STRING         AS tile,
        coordinates:su::INTEGER          AS su,
        coordinates:subcell::INTEGER     AS subcell,
        AVG(reading)                     AS avg_reading
    FROM data5035.spring26.sdg_001_ra226_scandata
    GROUP BY ALL
),

classified AS (
    -- Step 2: Apply the Ra-226 reference thresholds to label each subcell
    SELECT
        tile, su, subcell, avg_reading,
        CASE
            WHEN avg_reading < 5   THEN 'OK'       -- below action level
            WHEN avg_reading < 7.4 THEN 'Warning'  -- elevated; monitor closely
            ELSE                        'Alarm'    -- at or above remediation threshold
        END AS status
    FROM subcell_avgs
)

-- Step 3: Tile-level summary ordered by most severe first
SELECT
    tile,
    COUNT(*)                     AS total_subcells,
    COUNT_IF(status = 'OK')      AS ok_count,
    COUNT_IF(status = 'Warning') AS warning_count,
    COUNT_IF(status = 'Alarm')   AS alarm_count,
    ROUND(AVG(avg_reading), 4)   AS tile_avg_reading,
    ROUND(MAX(avg_reading), 4)   AS tile_max_reading
FROM classified
GROUP BY tile
ORDER BY alarm_count DESC, warning_count DESC;

## Part 4 — AI Remediation Prioritization

Use Snowflake Cortex `COMPLETE` to generate plain-language remediation recommendations for each tile, driven by its live statistics.

In [ ]:
%%sql -r dataframe_10
-- Use Snowflake Cortex COMPLETE to generate actionable remediation
-- recommendations for each tile based on its alarm/warning counts and
-- average/max Ra-226 reading.

WITH subcell_avgs AS (
    SELECT
        convert_xy(easting, northing)    AS coordinates,
        coordinates:tile::STRING         AS tile,
        coordinates:su::INTEGER          AS su,
        coordinates:subcell::INTEGER     AS subcell,
        AVG(reading)                     AS avg_reading
    FROM data5035.spring26.sdg_001_ra226_scandata
    GROUP BY ALL
),

classified AS (
    SELECT
        tile, su, subcell, avg_reading,
        CASE
            WHEN avg_reading < 5   THEN 'OK'
            WHEN avg_reading < 7.4 THEN 'Warning'
            ELSE                        'Alarm'
        END AS status
    FROM subcell_avgs
),

tile_summary AS (
    SELECT
        tile,
        COUNT(*)                     AS total_subcells,
        COUNT_IF(status = 'OK')      AS ok_count,
        COUNT_IF(status = 'Warning') AS warning_count,
        COUNT_IF(status = 'Alarm')   AS alarm_count,
        ROUND(AVG(avg_reading), 4)   AS tile_avg_reading,
        ROUND(MAX(avg_reading), 4)   AS tile_max_reading
    FROM classified
    GROUP BY tile
)

SELECT
    tile,
    alarm_count,
    warning_count,
    tile_avg_reading,
    tile_max_reading,
    SNOWFLAKE.CORTEX.COMPLETE(
        'mistral-large',
        CONCAT(
            'You are an environmental remediation specialist. ',
            'A rover survey measured Radium-226 levels (pCi/g) across grid tile ', tile, '. ',
            'Thresholds: OK < 5, Warning 5-7.4, Alarm >= 7.4. ',
            'Tile statistics: ',
              total_subcells, ' total subcells; ',
              ok_count,      ' OK; ',
              warning_count, ' Warning; ',
              alarm_count,   ' Alarm. ',
            'Average reading: ', tile_avg_reading, ' pCi/g. ',
            'Maximum reading: ', tile_max_reading, ' pCi/g. ',
            'In 2-3 sentences: assess the urgency and recommend immediate next steps.'
        )
    ) AS ai_recommendation
FROM tile_summary
ORDER BY alarm_count DESC, warning_count DESC;

---
## Feature 1 — Z-Score vs Global Mean (Assess)

**Formula:** `(tile_avg - global_avg) / global_stddev`

This feature contextualizes each tile's average Ra-226 reading against the **full dataset distribution**. A Z-score of +2 means a tile is two standard deviations above the dataset mean — a much richer signal than just the raw reading or threshold status alone.

**Why it's useful:** The raw threshold flags (OK / Warning / Alarm) tell you whether a tile is safe, but not *how unusual* it is relative to the rest of the survey area. The Z-score captures relative severity and is particularly useful for:
- Prioritizing within a set of "Warning" tiles
- Detecting tiles that are statistical outliers even if still technically below thresholds

In [ ]:
%%sql -r dataframe_zscore
-- Feature 1: Z-Score vs Global Mean
-- Compares each tile's average reading to the dataset-wide distribution.
-- z_score > 2 indicates a statistically significant elevation.

WITH subcell_avgs AS (
    -- Average raw readings down to the subcell grain first
    SELECT
        convert_xy(easting, northing)    AS coordinates,
        coordinates:tile::STRING         AS tile,
        coordinates:su::INTEGER          AS su,
        coordinates:subcell::INTEGER     AS subcell,
        AVG(reading)                     AS avg_reading
    FROM data5035.spring26.sdg_001_ra226_scandata
    GROUP BY ALL
),

tile_avgs AS (
    -- Roll up to tile-level average and max
    SELECT
        tile,
        AVG(avg_reading) AS tile_avg,
        MAX(avg_reading) AS tile_max
    FROM subcell_avgs
    GROUP BY tile
),

global_stats AS (
    -- Compute dataset-wide mean and standard deviation across all tile averages
    SELECT
        AVG(tile_avg) AS global_avg,
        STDDEV(tile_avg) AS global_stddev
    FROM tile_avgs
)

-- Join tile stats with global stats to compute the Z-score per tile.
-- NULLIF guards against division by zero if all tiles have identical averages.
SELECT
    t.tile,
    ROUND(t.tile_avg, 4)                                          AS tile_avg_reading,
    ROUND(t.tile_max, 4)                                          AS tile_max_reading,
    ROUND(g.global_avg, 4)                                        AS global_avg,
    ROUND(g.global_stddev, 4)                                     AS global_stddev,
    ROUND(
        (t.tile_avg - g.global_avg) / NULLIF(g.global_stddev, 0),
        4
    )                                                             AS z_score,
    -- Interpret the Z-score into a human-readable severity label
    CASE
        WHEN (t.tile_avg - g.global_avg) / NULLIF(g.global_stddev, 0) >= 2  THEN 'High Outlier'
        WHEN (t.tile_avg - g.global_avg) / NULLIF(g.global_stddev, 0) >= 1  THEN 'Above Average'
        WHEN (t.tile_avg - g.global_avg) / NULLIF(g.global_stddev, 0) >= -1 THEN 'Near Average'
        ELSE                                                                      'Below Average'
    END                                                           AS z_category
FROM tile_avgs   t
CROSS JOIN global_stats g
ORDER BY z_score DESC;

---
## Feature 2 — Sensor Variability Index (Combine)

**Formula:** `STDDEV(reading) / NULLIF(AVG(reading), 0)`

Also known as the **Coefficient of Variation (CV)**, this feature normalizes variability by the mean, making it comparable across tiles with very different average readings. A tile with a high CV has inconsistent readings — some spots are much hotter or colder than others within the same tile.

**Why it's useful:** Two tiles can have the same average reading but very different spread. A high CV suggests the tile may contain localized hotspots that are masked by averaging, warranting finer-grained follow-up inspection.

In [ ]:
%%sql -r dataframe_variability
-- Feature 2: Sensor Variability Index (Coefficient of Variation)
-- A high index means readings within the tile are inconsistent,
-- which may indicate localized hotspots masked by the tile average.

WITH subcell_avgs AS (
    -- Compute per-subcell averages as the base grain
    SELECT
        convert_xy(easting, northing)    AS coordinates,
        coordinates:tile::STRING         AS tile,
        coordinates:su::INTEGER          AS su,
        coordinates:subcell::INTEGER     AS subcell,
        AVG(reading)                     AS avg_reading
    FROM data5035.spring26.sdg_001_ra226_scandata
    GROUP BY ALL
)

SELECT
    tile,
    COUNT(*)                                                   AS subcell_count,
    ROUND(AVG(avg_reading), 4)                                 AS tile_avg,
    ROUND(STDDEV(avg_reading), 4)                              AS tile_stddev,
    ROUND(MAX(avg_reading), 4)                                 AS tile_max,
    -- CV = stddev / mean; NULLIF prevents division by zero on zero-mean tiles
    ROUND(
        STDDEV(avg_reading) / NULLIF(AVG(avg_reading), 0),
        4
    )                                                          AS variability_index,
    -- Flag tiles with high internal variability for deeper inspection
    CASE
        WHEN STDDEV(avg_reading) / NULLIF(AVG(avg_reading), 0) > 0.3 THEN 'High Variability'
        WHEN STDDEV(avg_reading) / NULLIF(AVG(avg_reading), 0) > 0.1 THEN 'Moderate Variability'
        ELSE                                                              'Stable'
    END                                                        AS variability_category
FROM subcell_avgs
GROUP BY tile
ORDER BY variability_index DESC NULLS LAST;

---
## Feature 3 — Local Rank Within Region (Rank)

**Description:** Rank each Survey Unit (SU) by its average Ra-226 reading **within its parent tile**, rather than globally across the entire survey area.

**Why it's useful:** A global ranking buries nuance — within any given tile, some SUs are much hotter than others. This local rank tells field teams which specific survey units to prioritize *within* each tile they are already investigating. Rank 1 = hottest SU in that tile.

In [ ]:
%%sql -r dataframe_local_rank
-- Feature 3: Local Rank of Survey Units Within Their Parent Tile
-- Rank 1 = hottest SU in that tile. Helps field teams prioritize
-- which specific areas to inspect within a tile they're already visiting.

WITH subcell_avgs AS (
    -- Average raw readings to the subcell grain
    SELECT
        convert_xy(easting, northing)    AS coordinates,
        coordinates:tile::STRING         AS tile,
        coordinates:su::INTEGER          AS su,
        coordinates:subcell::INTEGER     AS subcell,
        AVG(reading)                     AS avg_reading
    FROM data5035.spring26.sdg_001_ra226_scandata
    GROUP BY ALL
),

su_avgs AS (
    -- Roll up one level: average subcell readings to the SU grain
    SELECT
        tile,
        su,
        AVG(avg_reading) AS su_avg,
        MAX(avg_reading) AS su_max,
        COUNT(*)         AS subcell_count
    FROM subcell_avgs
    GROUP BY tile, su
)

SELECT
    tile,
    su,
    ROUND(su_avg, 4)  AS su_avg_reading,
    ROUND(su_max, 4)  AS su_max_reading,
    subcell_count,
    -- RANK() within each tile partition, ordered hottest-first
    -- RANK() leaves gaps after ties; use DENSE_RANK() to avoid gaps if preferred
    RANK() OVER (
        PARTITION BY tile          -- reset rank counter for each tile
        ORDER BY su_avg DESC       -- rank 1 = highest average reading in this tile
    )                             AS local_rank_in_tile,
    -- Also include a global rank for reference
    RANK() OVER (
        ORDER BY su_avg DESC
    )                             AS global_rank
FROM su_avgs
ORDER BY tile, local_rank_in_tile;